In [ ]:
# ============================
# 1. Install/import
# ============================
# In Colab, uncomment if needed:
# !pip install rasterio gcsfs geopandas fsspec pandas numpy scikit-learn tqdm

import os
import re
import json
import numpy as np
import pandas as pd
import rasterio
import gcsfs
from tqdm import tqdm
from datetime import timedelta
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix
)

In [ ]:
# ============================
# 2. User config
# ============================
GCS_BUCKET = "wildfire-detection-first"
GCS_PREFIX = "firms_daily_geotiff"   # folder inside bucket
REFERENCE_CSV = "/content/california_fire_perimeters.csv"
# must contain ALARM_DATE; CONT_DATE optional

OUTPUT_DAILY_FEATURES = "/content/firms_daily_features.csv"
OUTPUT_REFERENCE_DAYS = "/content/california_reference_fire_days.csv"
OUTPUT_THRESHOLD_RESULTS = "/content/firms_threshold_search_results.csv"
OUTPUT_BEST_DAYS = "/content/firms_best_threshold_predictions.csv"

# choose one:
TARGET_MODE = "active"   # "start" or "active"


In [ ]:
# ============================
# 3. Helpers
# ============================
date_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})")

def parse_date_from_path(path):
    m = date_pattern.search(path)
    if not m:
        return None
    return pd.to_datetime(m.group(1)).normalize()

def safe_to_datetime(series):
    # Specify the format to prevent UserWarning and improve parsing efficiency
    return pd.to_datetime(series, format='%m/%d/%Y %I:%M:%S %p', errors="coerce").dt.normalize()

def compute_binary_metrics(y_true, y_pred):
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "balanced_accuracy": bal_acc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# ============================
# 4. Read only 2020 FIRMS rasters from GCS
# ============================
import pandas as pd
import gcsfs

fs = gcsfs.GCSFileSystem(project=None)

all_blob_paths = sorted(fs.glob(f"{GCS_BUCKET}/{GCS_PREFIX}/*.tif"))
all_blob_paths = [f"gs://{p}" for p in all_blob_paths]

blob_paths = []
for path in all_blob_paths:
    dt = parse_date_from_path(path)
    if dt is not None and dt.year == 2020:
        blob_paths.append(path)

print(f"Found {len(blob_paths)} GeoTIFF files for 2020")
if blob_paths:
    print("First file:", blob_paths[0])
    print("Last file:", blob_paths[-1])

Found 366 GeoTIFF files for 2020
First file: gs://wildfire-detection-first/firms_daily_geotiff/2020-01-01.tif
Last file: gs://wildfire-detection-first/firms_daily_geotiff/2020-12-31.tif


In [ ]:
# ============================
# 5. Build daily FIRMS features
# ============================
rows = []

for path in tqdm(blob_paths):
    date = parse_date_from_path(path)
    if date is None:
        continue

    with rasterio.open(path) as src:
        arr = src.read()  # shape: (bands, H, W)

        band_names = list(src.descriptions) if src.descriptions else None
        if not band_names or all([b is None for b in band_names]):
            # fallback to your export order from notebook:
            # 1=firms_confidence, 2=firms_t21, 3=label
            band_names = ["firms_confidence", "firms_t21", "label"]

        band_map = {name: i + 1 for i, name in enumerate(band_names)}
        conf = arr[band_map["firms_confidence"] - 1].astype("float32")
        t21 = arr[band_map["firms_t21"] - 1].astype("float32")
        label = arr[band_map["label"] - 1].astype("float32")

        nodata = src.nodata
        if nodata is not None:
            valid_mask = np.ones(conf.shape, dtype=bool)
            valid_mask &= (conf != nodata) | (t21 != nodata) | (label != nodata)
        else:
            valid_mask = np.ones(conf.shape, dtype=bool)

        fire_mask = (label > 0) & valid_mask
        conf_fire = conf[fire_mask]
        t21_fire = t21[fire_mask]

        row = {
            "date": date,
            "n_valid_pixels": int(valid_mask.sum()),
            "n_fire_pixels_label": int(fire_mask.sum()),
        }

        if conf_fire.size == 0:
            row.update({
                "conf_max": 0.0,
                "conf_mean": 0.0,
                "conf_p90": 0.0,
                "t21_max": 0.0,
                "t21_mean": 0.0,
                "t21_p90": 0.0,
                "n_conf_ge_30": 0,
                "n_conf_ge_50": 0,
                "n_conf_ge_60": 0,
                "n_conf_ge_70": 0,
                "n_conf_ge_80": 0,
                "n_t21_ge_325": 0,
                "n_t21_ge_350": 0,
                "n_t21_ge_375": 0,
                "n_both_conf60_t21325": 0,
                "n_both_conf70_t21350": 0,
            })
        else:
            row.update({
                "conf_max": float(np.nanmax(conf_fire)),
                "conf_mean": float(np.nanmean(conf_fire)),
                "conf_p90": float(np.nanpercentile(conf_fire, 90)),
                "t21_max": float(np.nanmax(t21_fire)),
                "t21_mean": float(np.nanmean(t21_fire)),
                "t21_p90": float(np.nanpercentile(t21_fire, 90)),
                "n_conf_ge_30": int((conf_fire >= 30).sum()),
                "n_conf_ge_50": int((conf_fire >= 50).sum()),
                "n_conf_ge_60": int((conf_fire >= 60).sum()),
                "n_conf_ge_70": int((conf_fire >= 70).sum()),
                "n_conf_ge_80": int((conf_fire >= 80).sum()),
                "n_t21_ge_325": int((t21_fire >= 325).sum()),
                "n_t21_ge_350": int((t21_fire >= 350).sum()),
                "n_t21_ge_375": int((t21_fire >= 375).sum()),
                "n_both_conf60_t21325": int(((conf_fire >= 60) & (t21_fire >= 325)).sum()),
                "n_both_conf70_t21350": int(((conf_fire >= 70) & (t21_fire >= 350)).sum()),
            })

        rows.append(row)

daily_df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
daily_df.to_csv(OUTPUT_DAILY_FEATURES, index=False)
print(f"Saved {OUTPUT_DAILY_FEATURES}")

100%|██████████| 366/366 [13:57<00:00,  2.29s/it]

Saved /content/firms_daily_features.csv


In [ ]:
import urllib.request

REFERENCE_CSV = "/content/california_fire_perimeters.csv"
csv_url = "https://gis.data.cnra.ca.gov/api/download/v1/items/c3c10388e3b24cec8a954ba10458039d/csv?layers=0"

urllib.request.urlretrieve(csv_url, REFERENCE_CSV)
print("Downloaded:", REFERENCE_CSV)

ref = pd.read_csv(REFERENCE_CSV)
print(ref.head())

Downloaded: /content/california_fire_perimeters.csv
   OBJECTID    Year State Agency Unit ID  Fire Name Local Incident Number  \
0         1  2025.0    CA    USF     LPF    GIFFORD              00002181   
1         2  2025.0    CA    USF     LPF      MADRE              00001817   
2         3  2025.0    CA    USF     SNF     GARNET              00001684   
3         4  2025.0    CA    CDF     FKU       SALT              00018739   
4         5  2025.0    CA    CDF     LDF  PALISADES              00000738   

                                 IRWIN ID             Alarm Date  \
0  {08A6358F-8301-4093-9988-E265C8F6F0A9}   8/1/2025 12:00:00 AM   
1  {1AB682F7-CC0F-4DCE-87A4-FA4A9B67AB05}   7/2/2025 12:00:00 AM   
2  {68B988A1-C190-4F0C-9BEA-A658D389D167}  8/24/2025 12:00:00 AM   
3  {7F4A1474-6439-44F3-8F0D-083DCFD33639}   9/2/2025 12:00:00 AM   
4  {A7EA5D21-F882-44B8-BF64-44AB11059DC1}   1/7/2025 12:00:00 AM   

         Containment Date  ...  Management Objective  GIS Calculated Acres  

In [ ]:
# ============================
# 6. Build California reference fire-day labels
# ============================
ref = pd.read_csv(REFERENCE_CSV)

# Convert column names to uppercase and replace spaces with underscores
ref.columns = ref.columns.str.replace(' ', '_').str.upper()

if "ALARM_DATE" not in ref.columns:
    raise ValueError("REFERENCE_CSV must contain ALARM_DATE")

ref["ALARM_DATE"] = safe_to_datetime(ref["ALARM_DATE"])

if "CONT_DATE" in ref.columns:
    ref["CONT_DATE"] = safe_to_datetime(ref["CONT_DATE"])
else:
    ref["CONT_DATE"] = pd.NaT

ref = ref.dropna(subset=["ALARM_DATE"]).copy()
ref["CONT_DATE"] = ref["CONT_DATE"].fillna(ref["ALARM_DATE"])

all_days = pd.DataFrame({
    "date": pd.date_range(daily_df["date"].min(), daily_df["date"].max(), freq="D")
})

# fire start day
start_counts = ref.groupby("ALARM_DATE").size().rename("reference_start_count").reset_index()
start_counts = start_counts.rename(columns={"ALARM_DATE": "date"})
all_days = all_days.merge(start_counts, on="date", how="left")
all_days["reference_start_count"] = all_days["reference_start_count"].fillna(0).astype(int)
all_days["reference_start_day"] = (all_days["reference_start_count"] > 0).astype(int)

# fire active day
active_counter = {}
for _, r in tqdm(ref.iterrows(), total=len(ref)):
    s = r["ALARM_DATE"]
    e = r["CONT_DATE"]
    if pd.isna(s) or pd.isna(e):
        continue
    if e < s:
        e = s
    for d in pd.date_range(s, e, freq="D"):
        active_counter[d] = active_counter.get(d, 0) + 1

active_df = pd.DataFrame({
    "date": list(active_counter.keys()),
    "reference_active_count": list(active_counter.values())
})

all_days = all_days.merge(active_df, on="date", how="left")
all_days["reference_active_count"] = all_days["reference_active_count"].fillna(0).astype(int)
all_days["reference_active_day"] = (all_days["reference_active_count"] > 0).astype(int)

all_days.to_csv(OUTPUT_REFERENCE_DAYS, index=False)
print(f"Saved {OUTPUT_REFERENCE_DAYS}")

/tmp/ipykernel_1458/4291031118.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series, errors="coerce").dt.normalize()
100%|██████████| 17950/17950 [00:04<00:00, 3885.94it/s]

Saved /content/california_reference_fire_days.csv


In [ ]:
# ============================
# 7. Join FIRMS + reference
# ============================
df = daily_df.merge(all_days, on="date", how="left")

target_col = "reference_active_day" if TARGET_MODE == "active" else "reference_start_day"
df = df.dropna(subset=[target_col]).copy()
y_true = df[target_col].astype(int).values

In [ ]:
df.to_csv("/content/merged.csv", index=False)


In [ ]:
# ============================
# 8. Threshold search
# ============================
candidate_rules = []

# Single-feature rules
single_features = [
    "n_fire_pixels_label",
    "n_conf_ge_30",
    "n_conf_ge_50",
    "n_conf_ge_60",
    "n_conf_ge_70",
    "n_conf_ge_80",
    "n_t21_ge_325",
    "n_t21_ge_350",
    "n_t21_ge_375",
    "n_both_conf60_t21325",
    "n_both_conf70_t21350"
]

for feat in single_features:
    maxv = int(df[feat].max())
    for thr in range(1, max(2, min(maxv, 200)) + 1):
        y_pred = (df[feat] >= thr).astype(int).values
        m = compute_binary_metrics(y_true, y_pred)
        candidate_rules.append({
            "rule_type": "single",
            "feature_1": feat,
            "threshold_1": thr,
            "feature_2": None,
            "threshold_2": None,
            **m
        })

# Two-feature OR rules
pairs = [
    ("n_conf_ge_60", "n_t21_ge_325"),
    ("n_conf_ge_70", "n_t21_ge_350"),
    ("n_both_conf60_t21325", "n_fire_pixels_label"),
    ("n_both_conf70_t21350", "n_fire_pixels_label"),
]

for f1, f2 in pairs:
    max1 = int(df[f1].max())
    max2 = int(df[f2].max())
    for t1 in range(1, max(2, min(max1, 50)) + 1):
        for t2 in range(1, max(2, min(max2, 50)) + 1):
            y_pred = ((df[f1] >= t1) | (df[f2] >= t2)).astype(int).values
            m = compute_binary_metrics(y_true, y_pred)
            candidate_rules.append({
                "rule_type": "or",
                "feature_1": f1,
                "threshold_1": t1,
                "feature_2": f2,
                "threshold_2": t2,
                **m
            })

# Two-feature AND rules
for f1, f2 in pairs:
    max1 = int(df[f1].max())
    max2 = int(df[f2].max())
    for t1 in range(1, max(2, min(max1, 50)) + 1):
        for t2 in range(1, max(2, min(max2, 50)) + 1):
            y_pred = ((df[f1] >= t1) & (df[f2] >= t2)).astype(int).values
            m = compute_binary_metrics(y_true, y_pred)
            candidate_rules.append({
                "rule_type": "and",
                "feature_1": f1,
                "threshold_1": t1,
                "feature_2": f2,
                "threshold_2": t2,
                **m
            })

results = pd.DataFrame(candidate_rules)
results = results.sort_values(
    ["f1", "balanced_accuracy", "recall", "precision"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

results.to_csv(OUTPUT_THRESHOLD_RESULTS, index=False)
print(f"Saved {OUTPUT_THRESHOLD_RESULTS}")

best = results.iloc[0].to_dict()
print("\nBEST RULE")
print(json.dumps(best, indent=2, default=str))

Saved /content/firms_threshold_search_results.csv

BEST RULE
{
  "rule_type": "single",
  "feature_1": "n_t21_ge_325",
  "threshold_1": 6,
  "feature_2": null,
  "threshold_2": NaN,
  "precision": 0.5901060070671378,
  "recall": 0.8882978723404256,
  "f1": 0.7091295116772823,
  "balanced_accuracy": 0.6183062395409993,
  "tn": 62,
  "fp": 116,
  "fn": 21,
  "tp": 167
}


In [ ]:
# ============================
# 9. Apply best rule to all days
# ============================
if best["rule_type"] == "single":
    pred = (df[best["feature_1"]] >= best["threshold_1"]).astype(int)
elif best["rule_type"] == "or":
    pred = (
        (df[best["feature_1"]] >= best["threshold_1"]) |
        (df[best["feature_2"]] >= best["threshold_2"])
    ).astype(int)
else:
    pred = (
        (df[best["feature_1"]] >= best["threshold_1"]) &
        (df[best["feature_2"]] >= best["threshold_2"])
    ).astype(int)

df["predicted_fire_day_best"] = pred
df.to_csv(OUTPUT_BEST_DAYS, index=False)
print(f"Saved {OUTPUT_BEST_DAYS}")

Saved /content/firms_best_threshold_predictions.csv


In [ ]:
# ============================
# 10. Show top results
# ============================
print("\nTop 20 candidate rules:")
display(results.head(20))

print("\nBest-rule confusion counts:")
top_metrics = compute_binary_metrics(y_true, df["predicted_fire_day_best"].values)
print(top_metrics)


Top 20 candidate rules:


,rule_type,feature_1,threshold_1,feature_2,threshold_2,precision,recall,f1,balanced_accuracy,tn,fp,fn,tp
0,single,n_t21_ge_325,6,None,NaN,0.590106,0.888298,0.709130,0.618306,62,116,21,167
1,single,n_t21_ge_325,7,None,NaN,0.590106,0.888298,0.709130,0.618306,62,116,21,167
2,and,n_conf_ge_60,1,n_t21_ge_325,6.0,0.590747,0.882979,0.707889,0.618456,63,115,22,166
3,and,n_conf_ge_60,1,n_t21_ge_325,7.0,0.590747,0.882979,0.707889,0.618456,63,115,22,166
4,and,n_conf_ge_60,2,n_t21_ge_325,6.0,0.590747,0.882979,0.707889,0.618456,63,115,22,166
5,and,n_conf_ge_60,2,n_t21_ge_325,7.0,0.590747,0.882979,0.707889,0.618456,63,115,22,166
6,and,n_conf_ge_60,3,n_t21_ge_325,6.0,0.590747,0.882979,0.707889,0.618456,63,115,22,166
7,and,n_conf_ge_60,3,n_t21_ge_325,7.0,0.590747,0.882979,0.707889,0.618456,63,115,22,166
8,single,n_t21_ge_325,5,None,NaN,0.588028,0.888298,0.707627,0.615497,61,117,21,167
9,single,n_t21_ge_325,10,None,NaN,0.600000,0.861702,0.707424,0.627480,70,108,26,162



Best-rule confusion counts:
{'precision': 0.5901060070671378, 'recall': 0.8882978723404256, 'f1': 0.7091295116772823, 'balanced_accuracy': np.float64(0.6183062395409993), 'tn': np.int64(62), 'fp': np.int64(116), 'fn': np.int64(21), 'tp': np.int64(167)}
